In [1]:
!pip install pandas numpy scipy scikit-learn skorch matplotlib seaborn
!pip3 install torch --index-url https://download.pytorch.org/whl/cu128


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Looking in indexes: https://download.pytorch.org/whl/cu128

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import zscore
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, accuracy_score

from sklearn.metrics import (
    root_mean_squared_error, mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, 
    classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, TensorDataset, DataLoader
import itertools

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from datetime import datetime

In [3]:
orig_df = pd.read_csv('full_weather.csv', parse_dates=['datetime'])

def get_state_datapoint_counts(df):
    """
    Return a DataFrame of states and their respective number of datetime records,
    sorted from most to least.
    """
    counts = (
        df.groupby('state')['datetime']
          .count()
          .reset_index(name='datapoint_count')
          .sort_values(by='datapoint_count', ascending=False)
    )
    return counts

# Example usage:
state_counts = get_state_datapoint_counts(orig_df)
print(state_counts)

def get_city_datapoint_counts(df):
    """
    Return a DataFrame of states and their respective number of datetime records,
    sorted from most to least.
    """
    counts = (
        df.groupby('city')['datetime']
          .count()
          .reset_index(name='datapoint_count')
          .sort_values(by='datapoint_count', ascending=False)
    )
    return counts

# Example usage:
state_counts = get_city_datapoint_counts(orig_df)
print(state_counts)

def get_place_datapoint_counts(df):
    """
    Return a DataFrame of states and their respective number of datetime records,
    sorted from most to least.
    """
    counts = (
        df.groupby('place')['datetime']
          .count()
          .reset_index(name='datapoint_count')
          .sort_values(by='datapoint_count', ascending=False)
    )
    return counts

# Example usage:
state_counts = get_place_datapoint_counts(orig_df)
print(state_counts)

def plot_state_month_heatmap(df, agg_col='temperature', agg_func='mean', scale=None):
    # Ensure month column exists
    if 'month' not in df.columns:
        df['month'] = df['datetime'].dt.month
    
    # Create pivot table: state vs month
    pivot_df = df.pivot_table(
        values=agg_col,
        index='state',
        columns='month',
        aggfunc=agg_func
    )
    
    scale_factor = 1
    scale_label = ""
    if scale == 'k':
        scale_factor = 1e3
        scale_label = " (k)"
    elif scale == 'M':
        scale_factor = 1e6
        scale_label = " (M)"
    
    pivot_df = pivot_df / scale_factor

    # Plot heatmap
    plt.figure(figsize=(12, 8))
    sns.heatmap(pivot_df, annot=True, cmap="coolwarm", fmt=".2f")
    plt.title(f"Heatmap of {agg_func} {agg_col}{scale_label} by State and Month", fontsize=14)
    plt.xlabel("Month")
    plt.ylabel("State")
    plt.show()

plot_state_month_heatmap(df_filtered, agg_col='temperature', agg_func='mean')
plot_state_month_heatmap(df_filtered, agg_col='precipitation_total', agg_func='mean')


In [4]:
def filter_top_place_per_state(df, state_col='state', place_col='place', datetime_col='datetime'):
    """
    Keep only the place with the highest datetime count for each state.

    Parameters:
    df (DataFrame): Input dataframe.
    state_col (str): Column name for state.
    place_col (str): Column name for place.
    datetime_col (str): Column name for datetime.

    Returns:
    DataFrame: Filtered dataframe.
    """
    # Count unique datetime values per state-place
    counts = (
        df.groupby([state_col, place_col])[datetime_col]
        .nunique()
        .reset_index(name='datetime_count')
    )

    # Get the place with the max datetime_count per state
    top_places = counts.loc[counts.groupby(state_col)['datetime_count'].idxmax()]

    # Merge back to original df
    return df.merge(top_places[[state_col, place_col]], on=[state_col, place_col])

df_filtered = filter_top_place_per_state(orig_df, state_col='state', place_col='place', datetime_col='datetime')

## Pre-processing datasets

In [5]:
pre_df = df_filtered

# clear unnecessary columns
pre_df = pre_df.drop(columns=['pollutant_value'])

### Seasonal Interpolation of Missing Values
# Fill missing values per feature using monthly means.
pre_df['month'] = pre_df['datetime'].dt.month
features = [
    'pressure','dew_point','humidity','wind_speed',
    'gust','wind_chill','uv_index','feels_like_temperature',
    'visibility','solar_radiation']

month_means = pre_df.groupby('month')[features].mean()

# Fill NaNs by mapping month to mean values
for feat in features:
    na_mask = pre_df[feat].isna()
    pre_df.loc[na_mask, feat] = pre_df.loc[na_mask, 'month'].map(month_means[feat])

### Outlier Removal (Z‑score ±3)
# z_scores = df[features].apply(zscore)
# mask = (np.abs(z_scores) <= 3).all(axis=1)
# print(f"Removing {len(df) - mask.sum()} outliers.")
# df = df[mask].reset_index(drop=True)


### Feature Engineering
# - 1-hour, 1-day rolling average of precipitation and temperature
# - Monsoon month flag (Nov–Mar)
# - Coastal state flag
pre_df = pre_df.sort_values(['place','datetime']).reset_index(drop=True)

def fill_missing_precipitation(group):
    for col in ["precipitation_rate", "precipitation_total"]:
        # Mark where it rained
        rain_mask = group[col].fillna(0) > 0

        # Last rain datetime (ffill inside each place group)
        group["last_rain_time"] = group["datetime"].where(rain_mask).ffill()

        # Last rain value (ffill inside group)
        group["last_rain_value"] = group[col].where(rain_mask).ffill()

        # Missing rows
        missing_mask = group[col].isna()
        within_30min = (group["datetime"] - group["last_rain_time"]) <= pd.Timedelta("30min")

        # Fill with last rain value or 0
        group.loc[missing_mask & within_30min, col] = group.loc[missing_mask & within_30min, "last_rain_value"]
        group.loc[missing_mask & ~within_30min, col] = 0

        # Drop helper cols
        group = group.drop(columns=["last_rain_time", "last_rain_value"])

    return group

pre_df = pre_df.groupby("place", group_keys=False)[pre_df.columns].apply(fill_missing_precipitation)

# linear interpolation on missing temperature values
pre_df['temperature'] = pre_df['temperature'].interpolate(method='linear', limit_direction='both')

# 7‑day rolling avg per place
pre_df['precipitation_total_1h_avg'] = (
    pre_df.groupby("place")
      .rolling("1h", on="datetime")["precipitation_total"]
      .mean()
      .reset_index(drop=True)
)

pre_df['precipitation_total_1d_avg'] = (
    pre_df.groupby("place")
      .rolling("1d", on="datetime")["precipitation_total"]
      .mean()
      .reset_index(drop=True)
)

pre_df['precipitation_rate_1h_avg'] = (
    pre_df.groupby("place")
      .rolling("1h", on="datetime")["precipitation_rate"]
      .mean()
      .reset_index(drop=True)
)

pre_df['precipitation_rate_1d_avg'] = (
    pre_df.groupby("place")
      .rolling("1d", on="datetime")["precipitation_rate"]
      .mean()
      .reset_index(drop=True)
)

# 1‑hour rolling avg of temperature
pre_df['temperature_1h_avg'] = (
    pre_df.groupby("place")
      .rolling("1h", on="datetime")["temperature"]
      .mean()
      .reset_index(drop=True)
)

pre_df['temperature_1d_avg'] = (
    pre_df.groupby("place")
      .rolling("1d", on="datetime")["temperature"]
      .mean()
      .reset_index(drop=True)
)

# additional features
additional_features = [
    'temperature', 'precipitation_rate', 'precipitation_total',
    'precipitation_total_1h_avg', 'precipitation_total_1d_avg',
    'precipitation_rate_1h_avg', 'precipitation_rate_1d_avg',
    'temperature_1h_avg', 'temperature_1d_avg'
]

# Monsoon flag
pre_df['is_monsoon'] = pre_df['month'].apply(lambda m: 1 if m in [11,12,1,2,3] else 0)

# Coastal flag
coastal_states = ['Penang','Kedah','Selangor','Johor','Kelantan','Terengganu','Sarawak','Sabah']
pre_df['is_coastal'] = pre_df['state'].isin(coastal_states).astype(int)


### Rainfall flag
# if there is precipitation evaluated, it means there is rain
pre_df['rain_label'] = (pre_df['precipitation_rate'] > 0).astype(int)

### Scaling (Min‑Max)
scaler = MinMaxScaler()
cols_to_scale = features + additional_features

# Apply Min-Max scaling
scaled_values = scaler.fit_transform(pre_df[cols_to_scale])
pre_df[cols_to_scale] = scaled_values


## Temporal Split
Train: 1996–2013; Validation: 2014–2021; Test: 2022–present

In [6]:
pre_df['year'] = pre_df['datetime'].dt.year

train_df = pre_df[pre_df['year'].between(1996,2014)].copy()
val_df   = pre_df[pre_df['year'].between(2014,2021)].copy()
test_df  = pre_df[pre_df['year'] >= 2022].copy()
print(f"Train: {len(train_df)} rows | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 851893 rows | Val: 1952666 | Test: 2928737


## Temperature Linear Regression

In [7]:
class TemperatureLinearRegressionModel(torch.nn.Module):
    '''A simple linear regression model with one output.'''
    def __init__(self, input_dim, output_dim):
        super(TemperatureLinearRegressionModel, self).__init__()
        self.linear = torch.nn.Linear(input_dim, output_dim)

    def forward(self, x):
        y_pred = self.linear(x)
        return y_pred


def temperature_linear_train_val_dataset(features, labels, batch_size, num_workers=0):
    X_train = torch.tensor(train_df[features].values, dtype=torch.float32)
    y_train = torch.tensor(train_df[labels].values, dtype=torch.float32).unsqueeze(1)

    X_val = torch.tensor(val_df[features].values, dtype=torch.float32)
    y_val = torch.tensor(val_df[labels].values, dtype=torch.float32).unsqueeze(1)

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size, num_workers=num_workers)

    return train_loader, val_loader

def temperature_linear_test_dataset(features, labels, batch_size=128, num_workers=0):
    X_test = torch.tensor(test_df[features].values, dtype=torch.float32)
    y_test = torch.tensor(test_df[labels].values, dtype=torch.float32).unsqueeze(1)

    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, num_workers=num_workers)

    return test_loader

def train_temperature_linear_model(model, train_loader, val_loader, epochs, lr):

    history = {
        'train_loss': [],
        'train_rmse': [],
        'val_loss': [],
        'val_rmse': []
    }

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        start_time = datetime.now()

        model.train()
        train_loss, batch_rmse, train_total = 0, 0, 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            output = model(xb)
            loss = torch.sqrt(criterion(output, yb))
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            batch_rmse = torch.sqrt(torch.mean((output - yb) ** 2))
            train_total += batch_rmse.item()

        train_loss /= len(train_loader)
        train_rmse = train_total / len(train_loader)
        history['train_loss'].append(train_loss)
        history['train_rmse'].append(train_rmse)

        model.eval()
        val_loss, val_rmse, val_total = 0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)

                preds = model(xb)
                loss = torch.sqrt(criterion(preds, yb))

                val_loss += loss.item()

                batch_rmse = torch.sqrt(torch.mean((preds - yb) ** 2))
                val_total += batch_rmse.item()

        val_loss /= len(val_loader)
        val_rmse = val_total / len(val_loader)
        history['val_loss'].append(val_loss)
        history['val_rmse'].append(val_rmse)

        print(f"Epoch {epoch}/{epochs}, Train Loss: {train_loss:.4f}, Train RMSE: {train_rmse:.4f},  Val Loss: {val_loss:.4f}, Val RMSE: {val_rmse:.4f}")
        print(datetime.now() - start_time)

    return model, history
        
def evaluate_temperature_linear_model(model, test_loader):
    y_true = []
    y_pred = []

    model.eval()
    test_loss, test_rmse, test_total = 0, 0, 0

    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)

            outputs = model(xb)

            y_true.extend(yb.cpu().numpy())
            y_pred.extend(outputs.cpu().numpy())

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    print(f"Test MSE:  {mse:.4f}")
    print(f"Test RMSE: {rmse:.4f}")
    print(f"Test R2:  {r2:.4f}")


def plot_temperature_linear_model_history(history):
    epochs_range = range(1, len(history['train_loss']) + 1)

    plt.figure(figsize=(12, 5))

    # Plot Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, history['train_loss'], label='Training Loss')
    plt.plot(epochs_range, history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Loss over Epochs')
    plt.legend()
    plt.grid(True)

    # Plot Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, history['train_rmse'], label='Training Accuracy')
    plt.plot(epochs_range, history['val_rmse'], label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title('Accuracy over Epochs')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

In [8]:
def temperature_linear_model_gridsearch(features, labels, param_grid=None):
    '''Gridsearch over hyperparameters'''
    results = []
    best_model, best_params, best_result = None, None, None
    best_score = float("inf")  # lower RMSE is better

    for params in itertools.product(*param_grid.values()):
        param_dict = dict(zip(param_grid.keys(), params))

        train_loader, val_loader = temperature_linear_train_val_dataset(
            features, labels, 
            batch_size=param_dict['batch_size'], 
            num_workers=0
        )

        model = TemperatureLinearRegressionModel(
            input_dim=len(features), 
            output_dim=1
        ).to(device)

        model, model_result = train_temperature_linear_model(
            model, train_loader, val_loader, 
            epochs=param_dict['epochs'], 
            lr=param_dict['lr']
        )

        results.append((param_dict, model_result))
        print(f"Params: {param_dict}, Model Result: {model_result}")

        last_val_rmse = model_result['val_rmse'][-1]
        print(f"Params: {param_dict}, Last Val RMSE: {last_val_rmse:.4f}")

        # Keep only the best model
        if last_val_rmse < best_score:
            best_score = last_val_rmse
            best_params = param_dict
            best_result = model_result
            best_model = model  # keep this model


    # best_params, best_result = min(results, key=lambda x: min(x[1]['val_rmse']))
    print("\nBest Params:", best_params)
    print("Best Model Result:", best_result)

    # Plot the model loss
    plot_temperature_linear_model_history(best_result)

    # Evaluate on test set
    test_loader = temperature_linear_test_dataset(
        features, labels, 
        batch_size=max(param_grid['batch_size'], key=lambda x: x), 
        num_workers=0
    )

    evaluate_temperature_linear_model(best_model, test_loader)

## Rainfall binary classification

In [9]:
class RainBinaryClassifierModel(nn.Module):
    def __init__(self, input_size, hidden_size=64):
        super(RainBinaryClassifierModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)  # Single output neuron
        )
        
    def forward(self, x):
        return self.net(x)  # raw logits

def rainfall_binary_train_val_dataset(features, labels, batch_size=128, num_workers=0):
    X_train = torch.tensor(train_df[features].values, dtype=torch.float32)
    y_train = torch.tensor(train_df[labels].values, dtype=torch.float32).unsqueeze(1)

    X_val = torch.tensor(val_df[features].values, dtype=torch.float32)
    y_val = torch.tensor(val_df[labels].values, dtype=torch.float32).unsqueeze(1)

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size, num_workers=num_workers)

    return train_loader, val_loader

def rainfall_binary_test_dataset(features, labels, batch_size=128, num_workers=0):
    X_test = torch.tensor(test_df[features].values, dtype=torch.float32)
    y_test = torch.tensor(test_df[labels].values, dtype=torch.float32).unsqueeze(1)

    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, num_workers=num_workers)

    return test_loader

def train_rainfall_binary_model(model, train_loader, val_loader, epochs, lr):

    history = {
        'train_loss': [],
        'train_accuracy': [],
        'val_loss': [],
        'val_accuracy': []
    }

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        start_time = datetime.now()

        model.train()
        train_loss, train_correct, train_total = 0, 0, 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            outputs = model(xb)
            loss = criterion(outputs, yb)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * xb.size(0)

            # Predictions for accuracy
            predicted_labels = (torch.sigmoid(outputs) >= 0.5).float()
            train_correct += (predicted_labels == yb).sum().item()
            train_total += yb.size(0)

        train_loss /= train_total
        train_acc = train_correct / train_total
        history['train_loss'].append(train_loss)
        history['train_accuracy'].append(train_acc)

        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)

                outputs = model(xb)
                loss = criterion(outputs, yb)

                val_loss += loss.item() * xb.size(0)

                # Calculate accuracy
                predicted_labels = (torch.sigmoid(outputs) >= 0.5).float()
                val_correct += (predicted_labels == yb).sum().item()
                val_total += yb.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_acc)

        print(f"Epoch {epoch}/{epochs}, Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.4f},  Val Loss: {val_loss:.4f}, Val Accuracy: {val_acc:.4f}")
        print(datetime.now() - start_time)
    
    return model, history

def evaluate_rainfall_binary_model(model, test_loader):
    model.eval()  # evaluation mode
    y_preds = []
    y_labels = []
    test_correct,test_total = 0, 0

    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)

            outputs = model(xb)

            predicted_labels = (torch.sigmoid(outputs) >= 0.5).float()
            test_correct += (predicted_labels == yb).sum().item()
            test_total += yb.size(0)

            y_preds.extend(predicted_labels.cpu().numpy())
            y_labels.extend(yb.cpu().numpy())


    test_accuracy = 100 * test_correct / test_total

    print(f"Test Accuracy: {test_accuracy:.2f}%")

    print("Classification Report:")
    print(classification_report(y_labels, y_preds))

    accuracy = accuracy_score(y_labels, y_preds)
    f1 = f1_score(y_labels, y_preds)
    recall = recall_score(y_labels, y_preds)
    precision = precision_score(y_labels, y_preds)

    print(f"Accuracy: {accuracy:.2f}")
    print(f"F1 Score: {f1:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"Precision: {precision:.2f}")


def plot_rainfall_binary_model_history(history):
    epochs_range = range(1, len(history['train_loss']) + 1)

    plt.figure(figsize=(12, 5))

    # Plot Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, history['train_loss'], label='Training Loss')
    plt.plot(epochs_range, history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Loss over Epochs')
    plt.legend()
    plt.grid(True)

    # Plot Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, history['train_accuracy'], label='Training Accuracy')
    plt.plot(epochs_range, history['val_accuracy'], label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title('Accuracy over Epochs')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

In [10]:
def rainfall_binary_model_gridsearch(features, labels, param_grid=None):
    '''Gridsearch over hyperparameters'''
    results = []
    best_model, best_params, best_result = None, None, None
    best_accuracy = 0

    for params in itertools.product(*param_grid.values()):
        param_dict = dict(zip(param_grid.keys(), params))
        
        train_loader, val_loader = rainfall_binary_train_val_dataset(
            features, labels, 
            batch_size=param_dict['batch_size'], 
            num_workers=0
        )

        model = RainBinaryClassifierModel(
            input_size=len(features), 
        ).to(device)

        model, model_result = train_rainfall_binary_model(
            model, train_loader, val_loader, 
            epochs=param_dict['epochs'], 
            lr=param_dict['lr']
        )

        results.append((param_dict, model_result))
        print(f"Params: {param_dict}, Model Result: {model_result}")

        last_val_acc = model_result['val_accuracy'][-1]

        # Keep only the best model
        if last_val_acc > best_accuracy:
            best_accuracy = last_val_acc
            best_params = param_dict
            best_result = model_result
            best_model = model

    # best_params, best_result = min(results, key=lambda x: min(x[1]['val_rmse']))
    print("\nBest Params:", best_params)
    print("Best Model Result:", best_result)

    # Plot the model loss
    plot_rainfall_binary_model_history(model_result)

    # Evaluate on test set
    test_loader = rainfall_binary_test_dataset(
        features, labels, 
        batch_size=max(param_grid['batch_size'], key=lambda x: x), 
        num_workers=0
    )

    evaluate_rainfall_binary_model(best_model, test_loader)

## Long Short-Term Memory

In [11]:
class WeatherLSTMDataset(Dataset):
    def __init__(self, X, y_temp, y_rain, sequence_length=7):
        self.X = X
        self.y_temp = y_temp
        self.y_rain = y_rain
        self.seq_len = sequence_length

    def __len__(self):
        return len(self.X) - self.seq_len

    def __getitem__(self, idx):
        X_seq = self.X[idx:idx+self.seq_len]
        y_temp = self.y_temp[idx+self.seq_len]
        y_rain = self.y_rain[idx+self.seq_len]
        return X_seq, (y_temp, y_rain)


class WeatherLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super(WeatherLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0  # Dropout only works with >1 layer
        )
        self.dropout = nn.Dropout(dropout)
        self.fc_temp = nn.Linear(hidden_size, 1)
        self.fc_rain  = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]   # take last time step
        out = self.dropout(out)
        temp_out = self.fc_temp(out)   # regression (raw float)
        rain_out = self.fc_rain(out)   # classification (0–1 prob)

        return temp_out, rain_out  # return both outputs

def weather_lstm_train_val_dataset(features, label_1, label_2):
    X_train = torch.tensor(train_df[features].values, dtype=torch.float32)
    y_temp_train = torch.tensor(train_df[label_1].values, dtype=torch.float32).unsqueeze(1)
    y_rain_train = torch.tensor(train_df[label_2].values, dtype=torch.float32).unsqueeze(1)

    X_val = torch.tensor(val_df[features].values, dtype=torch.float32)
    y_temp_val = torch.tensor(val_df[label_1].values, dtype=torch.float32).unsqueeze(1)
    y_rain_val = torch.tensor(val_df[label_2].values, dtype=torch.float32).unsqueeze(1)

    return X_train, y_temp_train, y_rain_train, X_val, y_temp_val, y_rain_val

def weather_lstm_test_dataset(features, label_1, label_2, batch_size=128, num_workers=0):
    X_test = torch.tensor(test_df[features].values, dtype=torch.float32)
    y_temp_test = torch.tensor(test_df[label_1].values, dtype=torch.float32).unsqueeze(1)
    y_rain_test = torch.tensor(test_df[label_2].values, dtype=torch.float32).unsqueeze(1)

    test_loader = DataLoader(WeatherLSTMDataset(X_test, y_temp_test, y_rain_test), batch_size=batch_size, num_workers=num_workers)

    return test_loader

# train a single model for both temperature and rainfall prediction
def multitask_loss(temp_pred, temp_true, rain_pred, rain_true, mse_loss, bce_loss, alpha=0.5):
    loss_temp = torch.sqrt(mse_loss(temp_pred, temp_true))
    loss_rain = bce_loss(rain_pred, rain_true)
    return alpha * loss_temp + (1 - alpha) * loss_rain

def train_weather_lstm_model(model, train_loader, val_loader, epochs, lr):

    history = {
        'train_loss': [],
        'train_accuracy': [],
        'train_rmse': [],
        'val_loss': [],
        'val_accuracy': [],
        'val_rmse': []
    }

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    mse_loss = nn.MSELoss()
    bce_loss = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        model.train()
        train_loss, train_batch_rmse, train_rmse_total, train_correct, train_total, train_rmse = 0, 0, 0, 0, 0, 0

        for xb, (y_temp, y_rain) in train_loader:
            xb, y_temp, y_rain = xb.to(device), y_temp.to(device), y_rain.to(device)

            optimizer.zero_grad()
            temp_output, rain_output = model(xb)
            loss = multitask_loss(temp_output, y_temp, rain_output, y_rain, mse_loss, bce_loss)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            # Calculate RMSE for temperature
            train_batch_rmse = torch.sqrt(torch.mean((temp_output - y_temp) ** 2))
            train_rmse_total += train_batch_rmse.item()

            # Calculate accuracy for rainfall
            predicted_labels = (torch.sigmoid(rain_output) >= 0.5).float()
            train_correct += (predicted_labels == y_rain).sum().item()
            train_total += y_rain.size(0)

        train_loss /= train_total
        train_acc = train_correct / train_total
        train_rmse = train_rmse_total / train_total

        history['train_loss'].append(train_loss)
        history['train_accuracy'].append(train_acc)
        history['train_rmse'].append(train_rmse)

        # Validation
        model.eval()

        val_loss, val_batch_rmse, val_rmse_total, val_correct, val_total, val_rmse = 0, 0, 0, 0, 0, 0
        with torch.no_grad():
            for xb, (y_temp, y_rain) in val_loader:
                xb, y_temp, y_rain = xb.to(device), y_temp.to(device), y_rain.to(device)

                temp_output, rain_output = model(xb)
                loss = multitask_loss(temp_output, y_temp, rain_output, y_rain, mse_loss, bce_loss)

                val_loss += loss.item()

                # Calculate RMSE for temperature
                val_batch_rmse = torch.sqrt(torch.mean((temp_output - y_temp) ** 2))
                val_rmse_total += val_batch_rmse.item()

                # Calculate accuracy for rainfall
                predicted_labels = (torch.sigmoid(rain_output) >= 0.5).float()
                val_correct += (predicted_labels == y_rain).sum().item()
                val_total += y_rain.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total
        val_rmse = val_rmse_total / val_total
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_acc)
        history['val_rmse'].append(val_rmse)
        
        print(history)
        
        print(f"""\
            Epoch {epoch + 1}/{epochs}, 
            Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.4f}, Train RMSE: {train_rmse:.4f},
            Val Loss: {val_loss:.4f}, Val Accuracy: {val_acc:.4f}, Val RMSE: {val_rmse:.4f}
        """)

    return model, history

def evaluate_weather_lstm_model(model, test_loader):
    model.eval()  # evaluation mode
    y_temp_preds = []
    y_rain_preds = []
    y_temp_labels = []
    y_rain_labels = []
    
    test_correct,test_total = 0, 0
    
    with torch.no_grad():
        for xb, (y_temp, y_rain) in test_loader:
            xb, y_temp, y_rain = xb.to(device), y_temp.to(device), y_rain.to(device)

            temp_output, rain_output = model(xb)

            predicted_labels = (torch.sigmoid(rain_output) >= 0.5).float()
            test_correct += (predicted_labels == y_rain).sum().item()
            test_total += y_rain.size(0)

            y_rain_preds.extend(predicted_labels.cpu().numpy())
            y_rain_labels.extend(y_rain.cpu().numpy())

            y_temp_labels.extend(y_temp.cpu().numpy())
            y_temp_preds.extend(temp_output.cpu().numpy())


    mse = mean_squared_error(y_temp_labels, y_temp_preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_temp_labels, y_temp_preds)

    print(f"Test MSE:  {mse:.4f}")
    print(f"Test RMSE: {rmse:.4f}")
    print(f"Test R2:  {r2:.4f}")

    test_accuracy = 100 * test_correct / test_total

    print(f"Test Accuracy: {test_accuracy:.2f}%")

    print("Classification Report:")
    print(classification_report(y_rain_labels, y_rain_preds))

    accuracy = accuracy_score(y_rain_labels, y_rain_preds)
    f1 = f1_score(y_rain_labels, y_rain_preds)
    recall = recall_score(y_rain_labels, y_rain_preds)
    precision = precision_score(y_rain_labels, y_rain_preds)

    print(f"Accuracy: {accuracy:.2f}")
    print(f"F1 Score: {f1:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"Precision: {precision:.2f}")

def plot_rainfall_lstm_model_history(history):
    epochs_range = range(1, len(history['train_loss']) + 1)

    plt.figure(figsize=(12, 5))

    # Plot Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, history['train_loss'], label='Training Model Loss')
    plt.plot(epochs_range, history['val_loss'], label='Validation Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Loss over Epochs')
    plt.legend()
    plt.grid(True)

    # Plot Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, history['train_accuracy'], label='Training Rainfall Accuracy')
    plt.plot(epochs_range, history['val_accuracy'], label='Validation Rainfall Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title('Accuracy over Epochs')
    plt.legend()
    plt.grid(True)

    # Plot RMSE
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, history['train_rmse'], label='Training Temperature RMSE')
    plt.plot(epochs_range, history['val_rmse'], label='Validation Temperature RMSE')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Loss over Epochs')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

In [12]:
def weather_lstm_model_gridsearch(features, label_1, label_2, param_grid=None):
    
    results = {}
    best_model, best_params, best_result = None, None, None
    result_counter = 0

    # used to check if there is multiple runs
    num_runs = np.prod([len(v) for v in param_grid.values()])

    X_train, y_temp_train, y_rain_train, X_val, y_temp_val, y_rain_val = weather_lstm_train_val_dataset(features, label_1, label_2)

    for params in itertools.product(*param_grid.values()):
        result_counter += 1
        param_dict = dict(zip(param_grid.keys(), params))

        train_dataset = WeatherLSTMDataset(X_train, y_temp_train, y_rain_train, param_dict['sequence_length'])
        val_dataset = WeatherLSTMDataset(X_val, y_temp_val, y_rain_val, param_dict['sequence_length'])

        train_loader = DataLoader(train_dataset, batch_size=param_dict['batch_size'], shuffle=False) # no shuffle for LSTM
        val_loader = DataLoader(val_dataset, batch_size=param_dict['batch_size'], shuffle=False)

        model = WeatherLSTM(
            input_size=len(features), 
            hidden_size=param_dict['hidden_size'], 
            num_layers=param_dict['num_layers'], 
            dropout=param_dict['dropout']
        ).to(device)

        model, model_result = train_weather_lstm_model(
            model, train_loader, val_loader, 
            epochs=param_dict['epochs'], 
            lr=param_dict['lr']
        )
        
        results[result_counter] = {
            'param_dict': param_dict,
            'model_result': model_result,
            'model': model
        }

    if num_runs == 1:
        best_model = results[result_counter]['model']
        best_params = results[result_counter]['param_dict']
        best_result = results[result_counter]['model_result']
    else:
        # normalize rmse and accuracy for performance evaluation
        alpha = 0.5  # weight for accuracy vs RMSE

        acc_values = [results[result]['model_result']['val_accuracy'][-1] for result in results]
        acc_min, acc_max = min(acc_values), max(acc_values)
        if acc_min != acc_max:
            normalized_acc = [(x - acc_min) / (acc_max - acc_min) for x in acc_values]
        else:
            normalized_acc = [0] * len(acc_values)
            alpha = 0

        rmse_values = [results[result]['model_result']['val_rmse'][-1] for result in results]
        rmse_min, rmse_max = min(rmse_values), max(rmse_values)
        normalized_rmse = [(rmse_max - x) / (rmse_max - rmse_min) for x in rmse_values]
        if rmse_min != rmse_max:
            normalized_rmse = [(rmse_max - x) / (rmse_max - rmse_min) for x in rmse_values]
        else:
            normalized_rmse = [0] * len(rmse_values)
            alpha = 1

        for idx in range(len(results)):
            model_perf = alpha * normalized_acc[idx] + (1 - alpha) * normalized_rmse[idx]
            results[idx + 1]['performance'] = {
                'norm_acc': normalized_rmse[idx], 
                'norm_rmse': normalized_acc[idx], 
                'model_perf': model_perf
            }

        best_performance = max(results, key=lambda k: results[k]["performance"]["model_perf"])
        best_model = results[best_performance]['model']
        best_params = results[best_performance]['param_dict']
        best_result = results[best_performance]['model_result']

    # best_params, model_result = max(model_result['val_accuracy'], key=lambda x: x[1])
    print(f"Best Params: {best_params}")
    print(f"Best Model Result: {best_result}")

    # Plot the model loss
    plot_rainfall_lstm_model_history(model_result)

    # Evaluate on test set
    test_loader = weather_lstm_test_dataset(
        features, label_1, label_2,
        batch_size=max(param_grid['batch_size'], key=lambda x: x), 
        num_workers=0
    )
    
    evaluate_weather_lstm_model(best_model, test_loader)

## Random Forest

In [13]:
def weather_rf_train_val_test_dataset(features, label_1, label_2):
    X_train = train_df[features].values
    y_temp_train = train_df[label_1].values
    y_rain_train = train_df[label_2].values

    X_val = val_df[features].values
    y_temp_val = val_df[label_1].values
    y_rain_val = val_df[label_2].values

    X_test = test_df[features].values
    y_temp_test = test_df[label_1].values
    y_rain_test = test_df[label_2].values

    return X_train, y_temp_train, y_rain_train, X_val, y_temp_val, y_rain_val, X_test, y_temp_test, y_rain_test

def train_weather_rf_regressor_model(
        X_train, y_train_temp, 
        X_val, y_temp_val, 
        n_estimators, regressor_criterion, 
        max_depth, min_samples_split, min_samples_leaf, 
        max_samples, bootstrap):

    rf_temp = RandomForestRegressor(
        n_estimators = n_estimators,
        criterion=regressor_criterion,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_samples=max_samples,
        bootstrap=bootstrap,
        random_state = 42)
    
    rf_temp.fit(X_train, y_train_temp)

    val_preds_temp = rf_temp.predict(X_val)

    val_rmse = np.sqrt(mean_squared_error(y_temp_val, val_preds_temp))
    
    print(f"Validation RMSE (Temperature): {val_rmse:.4f}")

    return rf_temp, val_rmse

def train_weather_rf_classifier_model(
        X_train, y_train_rain, 
        X_val, y_rain_val, 
        n_estimators, classifier_criterion, 
        max_depth, min_samples_split, min_samples_leaf, 
        max_samples, bootstrap):

    rf_rain = RandomForestClassifier(
        n_estimators = n_estimators,
        criterion=classifier_criterion,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_samples=max_samples,
        bootstrap=bootstrap,
        random_state = 42)

    rf_rain.fit(X_train, y_train_rain)

    val_preds_rain = rf_rain.predict(X_val)

    val_acc  = accuracy_score(y_rain_val, val_preds_rain)
    
    print(f"Validation Accuracy (Rain): {val_acc:.4f}")

    return rf_rain, val_acc

def evaluate_weather_rf_model(rf_temp, rf_rain, X_test, y_temp_labels, y_rain_labels):
    y_temp_preds = rf_temp.predict(X_test)
    y_rain_preds = rf_rain.predict(X_test)

    mse = mean_squared_error(y_temp_labels, y_temp_preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_temp_labels, y_temp_preds)

    print(f"Test MSE:  {mse:.4f}")
    print(f"Test RMSE: {rmse:.4f}")
    print(f"Test R2:  {r2:.4f}")

    print("Classification Report:")
    print(classification_report(y_rain_labels, y_rain_preds))

    accuracy = accuracy_score(y_rain_labels, y_rain_preds)
    f1 = f1_score(y_rain_labels, y_rain_preds)
    recall = recall_score(y_rain_labels, y_rain_preds)
    precision = precision_score(y_rain_labels, y_rain_preds)

    print(f"Accuracy: {accuracy:.2f}")
    print(f"F1 Score: {f1:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"Precision: {precision:.2f}")

In [19]:
def weather_rf_model_gridsearch_not_used(features, label_1, label_2, param_grid=None):
    
    results = {}
    result_counter = 0
    best_model_temp, best_model_rain, best_params, best_performance = None, None, None, None
    best_performance = 0

    # used to check if there is multiple runs
    num_runs = np.prod([len(v) for v in param_grid.values()])

    X_train, y_temp_train, y_rain_train, X_val, y_temp_val, y_rain_val, X_test, y_temp_test, y_rain_test = weather_rf_train_val_test_dataset(features, label_1, label_2)

    for params in itertools.product(*param_grid.values()):
        param_dict = dict(zip(param_grid.keys(), params))

        rf_temp, rf_rain, val_rmse, val_acc = train_weather_rf_model(
            X_train, y_temp_train, y_rain_train,
            X_val, y_temp_val, y_rain_val,
            n_estimators=param_dict['n_estimators'],
            classifier_criterion=param_dict['classifier_criterion'],
            regressor_criterion=param_dict['regressor_criterion'],
            max_depth=param_dict['max_depth'],
            min_samples_split=param_dict['min_samples_split'],
            min_samples_leaf=param_dict['min_samples_leaf'],
            max_samples=param_dict['max_samples'],
            bootstrap=param_dict['bootstrap']
        )

        results[result_counter] = {
            'param_dict': param_dict,
            'val_rmse': val_rmse,
            'val_acc': val_acc,
            'model_temp': rf_temp,
            'model_rain': rf_rain
        }

    if num_runs == 1:
        best_model_temp = results[result_counter]['model_temp']
        best_model_rain = results[result_counter]['model_rain']
        best_params = results[result_counter]['param_dict']
    else:
        # normalize rmse and accuracy for performance evaluation
        alpha = 0.5  # weight for accuracy vs RMSE

        acc_values = [results[result]['val_acc'] for result in results]
        acc_min, acc_max = min(acc_values), max(acc_values)
        if acc_min != acc_max:
            normalized_acc = [(x - acc_min) / (acc_max - acc_min) for x in acc_values]
        else:
            normalized_acc = [0] * len(acc_values)
            alpha = 0

        rmse_values = [results[result]['val_rmse'] for result in results]
        rmse_min, rmse_max = min(rmse_values), max(rmse_values)
        normalized_rmse = [(rmse_max - x) / (rmse_max - rmse_min) for x in rmse_values]
        if rmse_min != rmse_max:
            normalized_rmse = [(rmse_max - x) / (rmse_max - rmse_min) for x in rmse_values]
        else:
            normalized_rmse = [0] * len(rmse_values)
            alpha = 1

        for idx in range(len(results)):
            model_perf = alpha * normalized_acc[idx] + (1 - alpha) * normalized_rmse[idx]
            results[idx + 1]['performance'] = {
                'norm_acc': normalized_rmse[idx], 
                'norm_rmse': normalized_acc[idx], 
                'model_perf': model_perf
            }

        best_performance = max(results, key=lambda k: results[k]["performance"]["model_perf"])

        best_model_temp = results[result_counter]['model_temp']
        best_model_rain = results[result_counter]['model_rain']
        best_params = results[result_counter]['param_dict']
        # best_result = results[best_performance]['val_rmse']

    print(f"Best Params: {best_params}")
    print(f"Best RF Val Acc: {results[best_performance]['val_acc']}")
    print(f"Best RF Val RMSE: {results[best_performance]['val_rmse']}")
    
    evaluate_weather_rf_model(best_model_temp, best_model_rain, X_test, y_temp_test, y_rain_test)

In [22]:
def weather_rf_model_gridsearch(features, label_1, label_2, regressor_param_grid, classifier_param_grid=None):
    
    regressor_results = {}
    classifier_results = {}
    result_counter = 0

    X_train, y_temp_train, y_rain_train, X_val, y_temp_val, y_rain_val, X_test, y_temp_test, y_rain_test = weather_rf_train_val_test_dataset(features, label_1, label_2)

    for params in itertools.product(*regressor_param_grid.values()):
        param_dict = dict(zip(regressor_param_grid.keys(), params))

        rf_temp, val_rmse = train_weather_rf_regressor_model(
            X_train, y_temp_train,
            X_val, y_temp_val,
            n_estimators=param_dict['n_estimators'],
            regressor_criterion=param_dict['regressor_criterion'],
            max_depth=param_dict['max_depth'],
            min_samples_split=param_dict['min_samples_split'],
            min_samples_leaf=param_dict['min_samples_leaf'],
            max_samples=param_dict['max_samples'],
            bootstrap=param_dict['bootstrap']
        )

        regressor_results[result_counter] = {
            'param_dict': param_dict,
            'val_rmse': val_rmse,
            'model_temp': rf_temp,
        }

    best_regressor_result = max(regressor_results, key=lambda k: regressor_results[k]['val_rmse'])
    best_model_temp = regressor_results[best_regressor_result]['model_temp']

    for params in itertools.product(*classifier_param_grid.values()):
        param_dict = dict(zip(classifier_param_grid.keys(), params))

        rf_rain, val_acc = train_weather_rf_classifier_model(
            X_train, y_rain_train,
            X_val, y_rain_val,
            n_estimators=param_dict['n_estimators'],
            classifier_criterion=param_dict['classifier_criterion'],
            max_depth=param_dict['max_depth'],
            min_samples_split=param_dict['min_samples_split'],
            min_samples_leaf=param_dict['min_samples_leaf'],
            max_samples=param_dict['max_samples'],
            bootstrap=param_dict['bootstrap']
        )

        classifier_results[result_counter] = {
            'param_dict': param_dict,
            'val_acc': val_acc,
            'model_rain': rf_rain
        }

    best_classifier_result = max(classifier_results, key=lambda k: classifier_results[k]['val_acc'])
    best_model_rain = classifier_results[best_classifier_result]['model_rain']

    print(f"Best RF Regressor Params: {regressor_results[best_regressor_result]['param_dict']}")
    print(f"Best RF Val Acc: {regressor_results[best_regressor_result]['val_rmse']}")
    print(f"Best RF Classifier Params: {classifier_results[best_classifier_result]['param_dict']}")
    print(f"Best RF Val RMSE: {classifier_results[best_classifier_result]['val_acc']}")
    
    evaluate_weather_rf_model(best_model_temp, best_model_rain, X_test, y_temp_test, y_rain_test)

## Ensemble method

In [16]:
def weather_ensemble_test_dataset(features, label_1, label_2):
    X_test = test_df[features].values
    y_temp_test = test_df[label_1].values
    y_rain_test = test_df[label_2].values

    return X_test, y_temp_test, y_rain_test

def evaluate_weather_ensemble_model(rf_temp, rf_rain, lstm_model, X_test, y_temp_labels, y_rain_labels):
    """
    rf_temp: trained RandomForestRegressor
    rf_rain: trained RandomForestClassifier
    lstm_model: trained LSTM model
    """

    preds_temp_rf = rf_temp.predict(X_test)
    preds_rain_rf = rf_rain.predict(X_test)
    
    X_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)

    lstm_model.eval()
    with torch.no_grad():
        preds_temp_lstm, preds_rain_lstm = lstm_model(X_tensor)
            
    preds_temp_lstm = preds_temp_lstm.cpu().numpy().flatten()
    preds_rain_lstm = (torch.sigmoid(preds_rain_lstm).cpu().numpy() > 0.5).astype(int)

    alpha = 0.5
    preds_temp_final = alpha * preds_temp_rf + (1 - alpha) * preds_temp_lstm

    preds_rain_prob = alpha * preds_rain_rf + (1 - alpha) * preds_rain_lstm
    preds_rain_final = (preds_rain_prob > 0.5).astype(int)

    
    mse = mean_squared_error(y_temp_labels, preds_temp_final)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_temp_labels, preds_temp_final)

    print(f"Test MSE:  {mse:.4f}")
    print(f"Test RMSE: {rmse:.4f}")
    print(f"Test R2:  {r2:.4f}")

    print("Classification Report:")
    print(classification_report(y_rain_labels, preds_rain_final))

    accuracy = accuracy_score(y_rain_labels, preds_rain_final)
    f1 = f1_score(y_rain_labels, preds_rain_final)
    recall = recall_score(y_rain_labels, preds_rain_final)
    precision = precision_score(y_rain_labels, preds_rain_final)

    print(f"Accuracy: {accuracy:.2f}")
    print(f"F1 Score: {f1:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"Precision: {precision:.2f}")



## Execution cell

In [23]:
### temperature regression
# temperature_features = [
#     'pressure','dew_point','humidity','wind_speed',
#     'gust','wind_chill','uv_index','feels_like_temperature',
#     'visibility','solar_radiation', 'precipitation_total',
#     'precipitation_total_1h_avg', 'precipitation_total_1d_avg',
#     'precipitation_rate_1h_avg', 'precipitation_rate_1d_avg',
#     'temperature_1h_avg', 'temperature_1d_avg', 'precipitation_rate'
# ]
# temperature_labels = 'temperature'

# temperature_linear_param_grid = {
#     'epochs'         : [2],
#     'batch_size'     : [128],
#     'lr'             : [0.001],
# }

# # temperature_binary_param_grid = {
# #     'epochs'         : [10, 20],
# #     'batch_size'     : [32],
# #     'lr'             : [0.001, 0.01],
# # }

# temperature_linear_model_gridsearch(temperature_features, temperature_labels, param_grid=temperature_linear_param_grid)

### rainfall binary classification
# rainfall_features = [
#     'pressure','dew_point','humidity','wind_speed',
#     'gust','wind_chill','uv_index','feels_like_temperature',
#     'visibility','solar_radiation', 'precipitation_total',
#     'precipitation_total_1h_avg', 'precipitation_total_1d_avg',
#     'precipitation_rate_1h_avg', 'precipitation_rate_1d_avg',
#     'temperature_1h_avg', 'temperature_1d_avg', 'temperature'
# ]

# rainfall_labels = 'rain_label'
# rainfall_binary_param_grid = {
#     'epochs'         : [5],
#     'batch_size'     : [128],
#     'lr'             : [0.001],
# }

# # rainfall_binary_param_grid = {
# #     'epochs'         : [10, 20],
# #     'batch_size'     : [32],
# #     'lr'             : [0.001, 0.01],
# # }

# rainfall_binary_model_gridsearch(rainfall_features, rainfall_labels, param_grid=rainfall_binary_param_grid)

### weather LSTM (temperature regression + rainfall classification)
# weather_lstm_features = [
#     'pressure','dew_point','humidity','wind_speed',
#     'gust','wind_chill','uv_index','feels_like_temperature',
#     'visibility','solar_radiation', 'precipitation_total',
#     'precipitation_total_1h_avg', 'precipitation_total_1d_avg',
#     'precipitation_rate_1h_avg', 'precipitation_rate_1d_avg',
#     'temperature_1h_avg', 'temperature_1d_avg'
# ]

# weather_lstm_param_grid = {
#     'epochs'          : [2],
#     'batch_size'      : [1024],
#     'lr'              : [0.001],
#     'sequence_length' : [7],
#     'hidden_size'     : [64],
#     'num_layers'      : [2],
#     'dropout'         : [0.1]
# }

# # weather_lstm_param_grid = {
# #     'epochs'         : [10, 20],
# #     'batch_size'     : [32],
# #     'lr'             : [0.001, 0.01],
# #     'sequence_length': [7, 14],
# #     'hidden_size'    : [32, 64],
# #     'num_layers'     : [1, 2],
# #     'dropout'        : [0.0, 0.2]
# # }

# weather_lstm_model_gridsearch(
#     features=weather_lstm_features, 
#     label_1='temperature',
#     label_2='rain_label',
#     param_grid=weather_lstm_param_grid)

### weather Random Forest (temperature regression + rainfall classification)
weather_rf_features = [

    'pressure','dew_point','humidity','wind_speed',
    'gust','wind_chill','uv_index','feels_like_temperature',
    'visibility','solar_radiation', 'precipitation_total',
    'precipitation_total_1h_avg', 'precipitation_total_1d_avg',
    'precipitation_rate_1h_avg', 'precipitation_rate_1d_avg',
    'temperature_1h_avg', 'temperature_1d_avg'
]

weather_rf_regressor_param_grid = {
    'n_estimators'          : [1000],
    'regressor_criterion'   : ["squared_error"],
    'max_depth'             : [None],
    'min_samples_split'     : [2],
    'min_samples_leaf'      : [1],
    'max_samples'           : [64],
    'bootstrap'             : [True] # Bagging to avoid data imbalance
}

weather_rf_classifier_param_grid = {
    'n_estimators'          : [1000],
    'classifier_criterion'  : ['gini'],
    'max_depth'             : [None],
    'min_samples_split'     : [2],
    'min_samples_leaf'      : [1],
    'max_samples'           : [64],
    'bootstrap'             : [True] # Bagging to avoid data imbalance
}




# weather_rf_param_grid = {
#     'n_estimators'          : [1000, 2000],
#     'classifier_criterion'  : ['gini', 'entropy', 'log_loss'],
#     'regressor_criterion'   : ["squared_error", "absolute_error", "friedman_mse", "poisson"],
#     'max_depth'             : [None],
#     'min_samples_split'     : [2, 5, 10],
#     'min_samples_leaf'      : [1, 2, 4],
#     'max_samples'           : [64],
#     'bootstrap'             : [True, False] # Bagging to avoid data imbalance
# }


weather_rf_model_gridsearch(
    features=weather_rf_features, 
    label_1='temperature',
    label_2='rain_label',
    regressor_param_grid=weather_rf_regressor_param_grid,
    classifier_param_grid=weather_rf_classifier_param_grid)

Validation RMSE (Temperature): 0.0203
Validation Accuracy (Rain): 0.9570
Best RF Regressor Params: {'n_estimators': 1000, 'regressor_criterion': 'squared_error', 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_samples': 64, 'bootstrap': True}
Best RF Val Acc: 0.02025038405115487
Best RF Classifier Params: {'n_estimators': 1000, 'classifier_criterion': 'gini', 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_samples': 64, 'bootstrap': True}
Best RF Val RMSE: 0.9569567965028325
Test MSE:  0.0004
Test RMSE: 0.0202
Test R2:  0.9329
Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97   2778115
           1       0.00      0.00      0.00    150622

    accuracy                           0.95   2928737
   macro avg       0.47      0.50      0.49   2928737
weighted avg       0.90      0.95      0.92   2928737



/home/han/uni/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/han/uni/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/han/uni/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Accuracy: 0.95
F1 Score: 0.00
Recall: 0.00
Precision: 0.00


/home/han/uni/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
